# 04b · Temporal graph model on REAL data (Global Fishing Watch, southern Indian EEZ) — 5 YEARS (2021–2025)

This notebook trains the temporal graph-attention model on the **real** data downloaded in notebook 02 (2021–2025, 5 years). It loads the data from your Google Drive (`MyDrive/india-fishing-gnn-data/real_gfw`), so nothing is downloaded again.

**Why "forecast next week"?** There are no public labels of confirmed illegal fishing for Indian waters. Training on invented labels would be dishonest, so the model learns a task real data *can* check:

> From everything seen up to week *t*, how likely is each vessel, in week *t + 1*, to have an AIS-off gap, an encounter (with any vessel, or with a carrier), loitering, fishing inside a marine protected area, or fishing inside an EEZ that is not its own flag state's (for example an Indian boat in Sri Lankan waters, or a foreign boat in Indian waters)?

The **investigation priority** is a fixed, documented combination of these calibrated forecasts. It is an analyst's weighting, not a learned "illegality" label.

**Held-out evidence.** None of the following is ever given to the model as input or target. The notebook then checks whether high-priority vessel-weeks are followed by more of it:
- GFW's own *possible intentional AIS disabling* flag on the next week's gaps;
- **Sentinel-1 radar** detections of vessels that match no AIS, within 20 km of the next week's gaps;
- **Sentinel-1 radar** detections of vessels that match no AIS, within 5 km of where the vessel fished next week, on the same day (unidentified boats operating alongside it).

**The graph**
- **Nodes:** vessel × week. Only fishing-related vessels are kept (fishing, carrier, support and bunker vessels, plus any vessel that met one of them); pure merchant traffic is dropped.
- **Edges:**
  - each vessel's previous 1–3 active weeks (past → present only);
  - real encounters in the same week;
  - co-location: vessels active in the same 0.25° cell in the same week.

**Comparisons**
- persistence ("next week looks like this week");
- gradient-boosted trees;
- the same network without a graph;
- the network with temporal edges only;
- the full graph.

**Splits**
- **Time split (main):** train on the past (2021 to Mar 2025), validate on Apr–Jun 2025, test on Jul–Dec 2025 (the validation and test windows are unchanged from the 2-year run, so scores stay comparable), with a one-week embargo at each boundary.
- **Vessel split:** test vessels are never seen in training.

**Runtime:** **use a GPU** (Runtime → Change runtime type → T4 GPU). The full run takes about 20–40 min; `FAST_MODE = True` takes about 5–10 min. A CPU run is possible with `FAST_MODE` but is slow (about 1 h or more).

> ⚠️ Real data, real limits. Many small Indian boats have no AIS at all. Scores rank **what to look at first**, never who is guilty. Vessel identities stay anonymised.

In [ ]:
# ---- Setup: works in Google Colab (CPU is enough) and locally --------------------
import os, sys, subprocess, pathlib, zipfile, glob
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Dan-droid-A-113/IUU-Fishing-GNN-India"   # clones the code; set to "" to upload the zip instead
USE_DRIVE = False      # True = keep project + outputs on Google Drive (restart-safe)

if IN_COLAB:
    PROJECT_DIR = pathlib.Path("/content/india-fishing-gnn")
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_DIR = pathlib.Path("/content/drive/MyDrive/india-fishing-gnn")
    if not (PROJECT_DIR / "src").exists():
        if REPO_URL:
            subprocess.run(["git", "clone", "-q", REPO_URL, str(PROJECT_DIR)], check=True)
        else:
            # accepts renamed downloads such as "india-fishing-gnn (1).zip"
            zips = sorted(glob.glob("/content/india-fishing-gnn*.zip"), key=os.path.getmtime)
            if not zips:
                from google.colab import files
                print("Project zip not found - choose india-fishing-gnn.zip from your computer:")
                uploaded = files.upload()   # file picker appears below this cell
                zips = [os.path.abspath(n) for n in uploaded if n.endswith(".zip")]
            if not zips:
                raise FileNotFoundError("No project zip uploaded.")
            print("Extracting", zips[-1])
            zipfile.ZipFile(zips[-1]).extractall(PROJECT_DIR.parent)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_DIR / "requirements-colab.txt")], check=True)
else:
    PROJECT_DIR = pathlib.Path.cwd()
    while not (PROJECT_DIR / "src").exists() and PROJECT_DIR != PROJECT_DIR.parent:
        PROJECT_DIR = PROJECT_DIR.parent

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print("Project directory:", PROJECT_DIR, "| Colab:", IN_COLAB)

## 0 · Mount Google Drive and find the saved real data

In [ ]:
from pathlib import Path
DRIVE_DATA = Path("/content/drive/MyDrive/india-fishing-gnn-data")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
from src.real_model import find_real_dir
REAL_DIR = find_real_dir()
OUT = (DRIVE_DATA / "outputs_04b") if IN_COLAB else (PROJECT_DIR / "outputs" / "real_04b")
(OUT / "figures").mkdir(parents=True, exist_ok=True)
print("real data:", REAL_DIR)
print("results will also be saved to:", OUT)
!ls -la {REAL_DIR}

## 1 · Settings

In [ ]:
import json, warnings, time
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
from src import real_model as rm
from src import visualization as viz
from src.config import FIGURE_DIR, METRIC_DIR

FAST_MODE = False            # True = 1 seed, time split only (about 5-10 min on a T4 GPU)
RUN_VESSEL_SPLIT = not FAST_MODE
DOWNLOAD_SAR_FOOTPRINT = True   # one-time (~10 min, needs GFW_API_TOKEN secret); cached on Drive afterwards
PROV = json.loads((REAL_DIR / "provenance.json").read_text()) if (REAL_DIR / "provenance.json").exists() else {}
START, END = PROV.get("start", "2021-01-01"), PROV.get("end", "2025-12-31")
BBOX = tuple(PROV.get("bbox_lat_lat_lon_lon", (5.0, 16.0, 70.0, 82.0)))
CFG = rm.RealConfig(seeds=(0,) if FAST_MODE else (0, 1, 2), study_start=START)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for p in (FIGURE_DIR, METRIC_DIR):
    p.mkdir(parents=True, exist_ok=True)
FIG = OUT / "figures"
print("device:", DEVICE, "| seeds:", CFG.seeds)
if DEVICE.type == "cpu":
    print("No GPU found. Runtime -> Change runtime type -> T4 GPU is strongly recommended.")
print(rm.REAL_MODEL_CAVEAT)

## 2a · Radar imaging footprint (one-time download)

Sentinel-1 does not photograph all of the open ocean every day. If there's no radar detection of an unidentified boat next to a vessel, that only means something where the satellite actually took a picture that day.

- **Imaging footprint:** this cell downloads radar detections of **all** vessels, including those matched to AIS. Any detection within 25 km on the same day proves the area was imaged.
- **Where it's used:** the radar evidence check in section 9 is evaluated only in those imaged places.
- **What it needs:** the `GFW_API_TOKEN` secret, as in notebook 02. The first run takes longer for 5 years (2024–2025 months are reused from the Drive cache; 2021–2023 download, roughly 15–20 min); after that the footprint is read from Drive.
- **Skipping it:** set `DOWNLOAD_SAR_FOOTPRINT = False`. The radar check is then reported without a coverage correction and is not reliable.

In [ ]:
SAR_ALL_FILE = REAL_DIR / f"sar_all_vessels_{START}_{END}.parquet"   # one file per study period
sar_all = None
if SAR_ALL_FILE.exists():
    sar_all = pd.read_parquet(SAR_ALL_FILE)
elif DOWNLOAD_SAR_FOOTPRINT:
    try:
        from src.gfw_real import get_token, make_client, fetch_sar, bbox_polygon
        client = make_client(get_token())
        cache = (DRIVE_DATA / "gfw_raw_cache") if IN_COLAB else None
        kw = {"cache_dir": cache} if cache else {}
        sar_all = await fetch_sar(client, bbox_polygon(BBOX), START, END, unmatched_only=False, **kw)
        sar_all.to_parquet(SAR_ALL_FILE, index=False)
    except Exception as ex:
        print("Footprint download failed - continuing without it:", type(ex).__name__, ex)
print("radar footprint:", f"{len(sar_all):,} cell-days" if sar_all is not None else "not available")

## 2 · Load and clean the real events

**Cleaning rules**
- A few GFW gap records span years (the raw maximum was about 48,000 h). These mean the vessel left the area or stopped transmitting for good, so gaps longer than **30 days** are not counted as gaps. They are kept only as a separate count.
- Single-event durations are capped at one week.
- Every encounter is given a row for **both** vessels.
- Events that started before the study period are dropped. These are a few very old port visits; without this step the weekly chart would start in 2014.

In [ ]:
ev_raw, sar, _ = rm.load_real(REAL_DIR)
print(f"events loaded: {len(ev_raw):,} | SAR cell-days (no AIS match): {len(sar):,} | "
      f"SAR cell-days (all vessels, imaging footprint): {len(sar_all) if sar_all is not None else 'not available'}")
ev, rep = rm.clean_events(ev_raw, CFG)
ev = rm.add_sar_evidence(ev, sar, CFG.sar_radius_km, sar_all=sar_all)   # held-out evidence only
fe = ev.event_type.eq("fishing")
rep["fishing events in radar-imaged areas (same day)"] = int((fe & ev.sar_imaged).sum())
rep["...of which with an unmatched SAR detection within 5 km"] = int((fe & ev.sar_imaged & (ev.sar_near_event > 0)).sum())
display(pd.Series(rep, name="value").to_frame())
f = ev[ev.event_type.eq("fishing")]
print("Fishing events by flag: share inside an EEZ that is not the flag state's (sanity check)")
display(f.assign(flag=f.flag.fillna("unknown")).groupby("flag").agg(
    fishing_events=("event_id", "size"), share_outside_flag_eez=("outside_flag_eez", "mean"))
    .sort_values("fishing_events", ascending=False).head(8).round(3))
g = ev[ev.gap_valid]
display(g.gap_h.describe(percentiles=[.5, .9, .99]).round(1).rename("valid gap hours").to_frame().T)

## 3 · Keep fishing-related vessels (drop pure merchant traffic)

In [ ]:
keep, vclass, sel = rm.select_vessels(ev, CFG)
display(sel)
ev = ev[ev.vessel_id.isin(keep)].reset_index(drop=True)
print(f"kept {len(keep):,} vessels and {len(ev):,} events")
display(ev.groupby("event_type").size().rename("events kept").to_frame())

## 4 · Vessel-week table: inputs, next-week targets, held-out evidence

In [ ]:
nodes = rm.weekly_table(ev, vclass, CFG)
print(f"vessel-weeks: {len(nodes):,} | with a next active week: {int(nodes.has_next.sum()):,}")
tcols = [f"y_{t}" for t in rm.TASKS]; ecols = [f"e_{k}" for k in rm.EVIDENCE]
rates = nodes[nodes.has_next][tcols + ecols].mean().rename("share of vessel-weeks (next week)")
display(rates.to_frame().style.format("{:.2%}"))
fig = viz.plot_weekly_activity(nodes, FIG / "real_weekly_activity.png",
                               split_lines={"validation starts": CFG.val_start, "test starts": CFG.test_start})
plt.show()

## 5 · Build the real temporal interaction graph

In [ ]:
t0 = time.time()
G = rm.build_real_graph(nodes, ev, CFG)
display(rm.graph_summary(G).rename("count").to_frame())
_E = G["edge_index"].shape[1]
_gb = _E * CFG.hidden * 4 * 4 * CFG.layers / 1e9   # rough: ~4 edge-sized float tensors per GAT layer kept for backprop
print(f"{G['n']:,} nodes, {_E:,} edges -> roughly {_gb:.0f} GB of GPU memory for training (a T4 has 15 GB).")
if DEVICE.type == "cuda":
    _tot = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}, {_tot:.0f} GB")
    if _gb > 0.8 * _tot:
        print("WARNING: likely out of memory. Lower CFG.max_total_edges (e.g. 3_000_000), CFG.temporal_lags to 2, or CFG.hidden to 32, then re-run this cell.")
print(f"built in {time.time() - t0:.0f}s | edge features: {rm.EDGE_COLS} + 8-d learned time encoding (lag in weeks)")

## 6 · Leakage-safe splits

In [ ]:
splits = {"time": rm.time_split(nodes, CFG)}
if RUN_VESSEL_SPLIT:
    splits["vessel"] = rm.vessel_split(nodes)
for name, (sp, gtrain) in splits.items():
    t = pd.DataFrame({"split": sp, "has_next": nodes.has_next})
    for c in tcols:
        t[c] = nodes[c] * nodes.has_next
    tab = t.groupby("split").agg(nodes=("has_next", "size"), with_target=("has_next", "sum"),
                                 **{c.replace("y_", "pos_"): (c, "sum") for c in tcols})
    print(f"--- {name} split")
    display(tab.astype(int))
    if name == "vessel":
        display(rm.cross_split_edges(G, sp).rename("count").to_frame())

## 7 · Train and compare

- Every neural model is trained **inductively**: while fitting, it sees only the training part of the graph.
- Early stopping uses the validation macro PR-AUC.
- Forecasts are calibrated on validation (Platt scaling) before they are combined into the priority.

In [ ]:
t0 = time.time()
res = rm.run_real_experiment(G, splits, CFG, DEVICE)
print(f"total training time: {(time.time() - t0) / 60:.1f} min")
runs, tsum, esum = rm.summarize(res)
for df, name in ((res["runs"], "runs"), (res["tasks"], "task_metrics"), (res["evidence"], "evidence_metrics")):
    df.to_csv(OUT / f"real_{name}.csv", index=False); df.to_csv(METRIC_DIR / f"real_{name}.csv", index=False)
display(runs.round(4))

## 8 · Forecast quality per behaviour (test set)

In [ ]:
for sp in splits:
    display(tsum[tsum.split == sp].pivot(index="task", columns="model", values="pr_auc").round(3)
            .assign(base_rate=tsum[tsum.split == sp].groupby("task").base_rate.first().round(3)))
    fig = viz.plot_real_task_scores(tsum, sp, FIG / f"real_task_pr_auc_{sp}.png"); plt.show()
    fig = viz.plot_real_training(res["histories"], sp, FIG / f"real_training_{sp}.png"); plt.show()

## 9 · Does the priority point at held-out evidence?

- **Lift in the top 5%:** how many times more common the evidence is among the top 5% of vessel-weeks than on average. A lift of 1 means no better than random.
- **Activity-only reference:** a vessel that simply has many events will also have more of everything. The *Activity only* row shows how much of the lift comes from that alone, and a useful model must beat it.

In [ ]:
for sp in splits:
    e = esum[esum.split == sp]
    display(e.pivot(index="evidence", columns="model", values="lift_top5pct").round(2))
    display(e.pivot(index="evidence", columns="model", values="pr_auc").round(3)
            .assign(base_rate=e.groupby("evidence").base_rate.first().round(4)))
    fig = viz.plot_evidence_lift(esum, sp, FIG / f"real_evidence_lift_{sp}.png"); plt.show()

## 10 · Does the graph help?

- The full graph is compared against the *same* network without a graph, over all test vessel-weeks.
- It is also compared on the subgroup where the graph carries information: vessel-weeks with a real encounter or co-located vessels.

In [ ]:
s, d = G["edge_index"]; et = G["edge_type"]
has_social = np.zeros(G["n"], bool); has_social[d[et > 0]] = True
rows = []
for sp_name, (sp, _) in splits.items():
    for model in ("Temporal GAT (full graph)", "Temporal GAT (temporal edges only)", "Same model, no graph (MLP)",
                  "Gradient boosting (no graph)"):
        for (spn, m, seed), p in res["preds"].items():
            if spn != sp_name or m != model:
                continue
            for grp_name, grp in (("all test vessel-weeks", np.ones(G["n"], bool)), ("with encounter / co-location neighbours", has_social)):
                sub = np.where(grp, sp, "other")
                tm = rm.task_metrics(p["P"], G, sub, p["tasks"])
                rows.append({"split": sp_name, "model": model, "seed": seed, "group": grp_name, "macro_pr_auc": tm.pr_auc.mean()})
gh = pd.DataFrame(rows).groupby(["split", "group", "model"]).macro_pr_auc.agg(["mean", "std"]).round(4)
gh.to_csv(OUT / "real_graph_help.csv"); display(gh)

## 11 · Investigation-priority alerts (test weeks, time split)

In [ ]:
best = rm.best_seed(res, "time", "Temporal GAT (full graph)")
alerts = rm.alert_table(G, best, splits["time"][0], top=50, weights=CFG.priority_weights)
rm.alerts_json(alerts, OUT / "real_alerts_top50.json",
               {"model": "Temporal GAT (full graph)", "split": "time", "period": f"test weeks from {CFG.test_start}",
                "priority_weights": CFG.priority_weights})
alerts.to_csv(OUT / "real_alerts_top50.csv", index=False)
show = ["rank", "vessel_id", "vessel_class", "week", "lat", "lon", "priority", "main_driver",
        "this_week_n_gaps", "this_week_n_enc", "next_week_observed"] + [c for c in alerts if c.startswith("heldout_")]
display(alerts[show].head(25))
print("share of top-50 whose next week showed held-out evidence:",
      alerts[[c for c in alerts if c.startswith("heldout_")]].any(axis=1).mean().round(3))

In [ ]:
from src.india_layers import get_layers
prov = json.loads((REAL_DIR / "provenance.json").read_text()) if (REAL_DIR / "provenance.json").exists() else {}
bbox = tuple(prov.get("bbox_lat_lat_lon_lon", (5.0, 16.0, 70.0, 82.0)))
test_weeks = nodes.week >= pd.Timestamp(CFG.test_start)
fg = ev[ev.gap_valid & ev.vclass.eq("fishing") & (ev.week >= pd.Timestamp(CFG.test_start))]
fg = fg.dropna(subset=["off_lat"]) if "off_lat" in fg else fg.iloc[:0]
fig = viz.plot_real_alerts_map(nodes[test_weeks], alerts, fg, bbox, layers=get_layers("IN-GOM"),
                               title=f"Top-50 investigation priorities and fishing-vessel AIS-off positions (test weeks from {CFG.test_start})",
                               path=FIG / "real_alerts_map.png")
plt.show()

## 12 · Save summary (to the project and to Drive)

In [ ]:
import shutil
summary = {
    "notebook": "04b_real_graph_model", "data": "Global Fishing Watch (real)", "caveat": rm.REAL_MODEL_CAVEAT,
    "device": str(DEVICE), "config": {k: v for k, v in vars(CFG).items()}, "cleaning": rep,
    "graph": rm.graph_summary(G).to_dict(),
    "models_mean": runs.to_dict("records"),
    "tasks_mean": tsum.to_dict("records"),
    "evidence_mean": esum.to_dict("records"),
}
(OUT / "real_model_summary.json").write_text(json.dumps(summary, indent=2, default=str))
shutil.copy(OUT / "real_model_summary.json", METRIC_DIR / "real_model_summary.json")
for f in FIG.glob("real_*.png"):
    shutil.copy(f, FIGURE_DIR / f.name)
print("saved:", *sorted(p.name for p in OUT.glob("*.*")), sep="\n  ")

## How to read the results

**Questions the tables answer**
1. **Can real next-week behaviour be forecast at all?** Compare each model's PR-AUC with the base rate (the black tick). Persistence is the bar to beat: many vessels simply repeat what they did last week.
2. **Does the graph add anything?** Compare *Temporal GAT (full graph)* with the *no graph* and *temporal edges only* versions, overall and on vessel-weeks with neighbours (section 10). If they tie, the honest conclusion is that the encounter and co-location structure adds little beyond each vessel's own history on this data.
3. **Does the priority point at independent evidence?** Section 9 answers this. A lift above the *Activity only* reference means the model finds more than just the busiest vessels.

**Can conclude:** how well real vessel behaviour in the southern Indian EEZ can be forecast from GFW events, whether graph structure helps, and whether the resulting priority is enriched for independent radar and GFW risk evidence.

**Cannot conclude:**
- that any vessel acted illegally: gaps, encounters and fishing in a neighbouring EEZ all have lawful explanations (coverage, transhipment licences, access agreements);
- anything about boats without AIS, except indirectly through radar;
- that attention weights explain decisions. Milestone 4 adds proper explanations (GNNExplainer, SHAP, counterfactuals) for these alerts.

Attribution: data from **Global Fishing Watch**, non-commercial use.